# Muzen Python Review

Run a local review through the Python SDK preview and inspect replayed events, the final result, and redacted artifacts.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "sdk" / "python" / "muzen").exists():
            return path
    raise RuntimeError("could not find repository root containing sdk/python/muzen")

repo_root = find_repo_root(Path.cwd())
sdk_path = repo_root / "sdk" / "python"
if str(sdk_path) not in sys.path:
    sys.path.insert(0, str(sdk_path))

from muzen import Client, ReviewOptions, local

runner_path = os.environ.get("MUZEN_RUNNER_PATH") or str(repo_root / "target" / "debug" / "muzen-runner")
runner_path

In [ ]:
client = await Client.create(runner_path=runner_path)
review = await client.review(local(str(repo_root)), ReviewOptions(scope_files=["Cargo.toml"]))
review.id, review.status

In [ ]:
events = [event async for event in review.events()]
[(event.cursor, event.type) for event in events]

In [ ]:
result = await review.wait()
{
    "conclusion": result.conclusion,
    "summary": result.summary,
    "findings": len(result.findings),
    "files_reviewed": result.coverage.files_reviewed,
}

In [ ]:
artifacts = await review.export_artifacts()
{
    "artifact_count": artifacts.artifact_count,
    "total_bytes": artifacts.total_bytes,
}

In [ ]:
await client.close()